# Grain Portfolio Backtest Research

This is a portfolio construction check on the same grain data. It compares a simple shared-score outright book with cross-grain spread legs across corn, soybean, and wheat.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from grain_backtest_core import load_train_set
from shared_backtest import (
    active_metrics,
    backtest_positions_with_costs,
    build_feature_panels,
    clean_signal,
    named_period_check as shared_named_period_check,
    positions_from_signal,
)
from research_config import DEFAULT_MARGIN_PER_LOT, SPLIT_DATE

DATA_DIR = "train_set"
OOS_START = pd.Timestamp(SPLIT_DATE)
DD_CAPITAL_USD = 10_000.0
BOOK_COLUMNS = ["CORN", "SOYABEAN", "WHEAT_SRW"]

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


## 1. Data And Book Helpers

Use the same train set, costs, and margin assumptions as the individual notebooks. Portfolio books are built from positions first, then backtested once as a combined book.


In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl = build_feature_panels(data)
trading_index = futures_pnl.index
feature_panels = {name: panel.reindex(trading_index).fillna(0.0) for name, panel in feature_panels.items()}


def combine_positions(frames, columns=BOOK_COLUMNS):
    out = pd.DataFrame(0.0, index=trading_index, columns=columns)
    for frame in frames:
        aligned = frame.reindex(trading_index).reindex(columns=columns, fill_value=0.0).fillna(0.0)
        out = out.add(aligned, fill_value=0.0)
    return out.fillna(0.0)


def backtest_book(positions, columns=None):
    columns = list(positions.columns if columns is None else columns)
    aligned = positions.reindex(trading_index).reindex(columns=columns, fill_value=0.0).fillna(0.0)
    return backtest_positions_with_costs(
        aligned,
        futures_pnl[columns],
        trade_cost_per_lot=8.75,
        holding_cost_rate=0.05,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )[0]


def metric_row(book, bt):
    oos = active_metrics(bt, bt.index >= OOS_START)
    full = active_metrics(bt)
    return {
        "book": book,
        "oos_sharpe": oos["sharpe"],
        "oos_pnl": oos["total_pnl"],
        "oos_dd_pct": 100.0 * oos["max_drawdown"] / DD_CAPITAL_USD,
        "oos_active_days": oos["days"],
        "full_sharpe": full["sharpe"],
        "full_pnl": full["total_pnl"],
        "full_dd_pct": 100.0 * full["max_drawdown"] / DD_CAPITAL_USD,
        "turnover": full["turnover"],
        "avg_gross_exposure": full["avg_gross_exposure"],
    }


def display_table(table):
    display(table.round(3))


raw_corr = futures_pnl[BOOK_COLUMNS].loc[trading_index >= OOS_START].corr()
display(Markdown("### OOS raw futures PnL correlation"))
display_table(raw_corr)


### OOS raw futures PnL correlation

,CORN,SOYABEAN,WHEAT_SRW
CORN,1.000,0.591,0.558
SOYABEAN,0.591,1.000,0.377
WHEAT_SRW,0.558,0.377,1.000


## 2. Outright And Spread Legs

Use one common score recipe for the three markets, then compare outright legs, score-difference spreads, and the combined book. A positive spread score means long the first product and short the second product.


In [3]:
def market_score(commodity):
    panel = feature_panels[commodity]
    price = (panel["mom_20"] + panel["mom_60"] + panel["rev_5"]) / 3.0
    curve = (panel["curve_spread"] + panel["curve_ratio"] + panel["curve_change_20"]) / 3.0
    physical = (
        -panel["public_inventory_change"]
        - panel["receipts_change"]
        - panel["cgl_inventory_change"]
        + panel["crush_surprise"]
        + panel["crush_utilization"]
    ) / 5.0
    cot = (
        panel["cot_mm_level"]
        + panel["cot_mm_change"]
        + panel["cot_pm_oi_level"]
        + panel["cot_pm_oi_change"]
    ) / 4.0
    return clean_signal(0.45 * price + 0.20 * curve + 0.20 * physical + 0.15 * cot, trading_index)


def outright_positions(commodity, score):
    return positions_from_signal(
        score,
        futures_pnl[[commodity]],
        commodity,
        target_daily_vol=50.0,
        max_abs_lot=0.30,
        halflife=4.0,
        threshold=0.10,
    )


def spread_positions(first, second, score, target_daily_pair_vol=35.0, max_leg_lot=0.25):
    score = clean_signal(score, trading_index)
    score = pd.Series(np.tanh(score / 2.0), index=trading_index).ewm(
        halflife=5.0,
        adjust=False,
        min_periods=1,
    ).mean()
    score[score.abs() < 0.15] = 0.0

    vol = futures_pnl[[first, second]].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    positions = pd.DataFrame(0.0, index=trading_index, columns=[first, second])
    positions[first] = (score * target_daily_pair_vol / vol[first]).clip(-max_leg_lot, max_leg_lot)
    positions[second] = (-score * target_daily_pair_vol / vol[second]).clip(-max_leg_lot, max_leg_lot)

    rebalance = pd.Series(False, index=trading_index)
    rebalance.iloc[::5] = True
    return positions.fillna(0.0).where(rebalance).ffill().fillna(0.0)


scores = {
    "corn": market_score("CORN"),
    "soybean": market_score("SOYABEAN"),
    "wheat": market_score("WHEAT_SRW"),
}
commodities = {"corn": "CORN", "soybean": "SOYABEAN", "wheat": "WHEAT_SRW"}

outright = {
    name: outright_positions(commodity, scores[name])
    for name, commodity in commodities.items()
}
spreads = {
    "corn_soybean_spread": spread_positions("CORN", "SOYABEAN", scores["corn"] - scores["soybean"]),
    "corn_wheat_spread": spread_positions("CORN", "WHEAT_SRW", scores["corn"] - scores["wheat"]),
    "soybean_wheat_spread": spread_positions("SOYABEAN", "WHEAT_SRW", scores["soybean"] - scores["wheat"]),
}

outright_backtests = {name: backtest_book(pos, pos.columns) for name, pos in outright.items()}
spread_backtests = {name: backtest_book(pos, pos.columns) for name, pos in spreads.items()}
outright_book = combine_positions(outright.values())
spread_book = combine_positions(spreads.values())
combined_book = combine_positions([outright_book, spread_book]).clip(-0.75, 0.75)

outright_bt = backtest_book(outright_book)
spread_bt = backtest_book(spread_book)
combined_bt = backtest_book(combined_book)
strategy_pnl = pd.DataFrame({
    **{name: bt["net_pnl"] for name, bt in outright_backtests.items()},
    "spread_book": spread_bt["net_pnl"],
})

outright_results = pd.DataFrame(
    [metric_row(name, bt) for name, bt in outright_backtests.items()]
    + [metric_row("outright_book", outright_bt)]
)
spread_results = pd.DataFrame(
    [metric_row(name, bt) for name, bt in spread_backtests.items()]
    + [metric_row("spread_book", spread_bt)]
)
portfolio_results = pd.DataFrame([
    metric_row("outright_book", outright_bt),
    metric_row("spread_book", spread_bt),
    metric_row("outright_plus_spreads", combined_bt),
])


display(Markdown("### Outright legs"))
display_table(outright_results)
display(Markdown("### Spread legs"))
display_table(spread_results)
display(Markdown("### Portfolio comparison"))
display_table(portfolio_results)
display(Markdown("### OOS strategy PnL correlation"))
display_table(strategy_pnl.loc[trading_index >= OOS_START].corr())


### Outright legs

,book,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,turnover,avg_gross_exposure
0,corn,0.137,51.875,-3.101,471.000,0.182,219.355,-3.524,0.003,0.036
1,soybean,0.797,308.313,-3.251,397.000,0.345,432.415,-4.326,0.001,0.017
2,wheat,-0.958,-309.896,-3.950,461.000,-0.413,-498.905,-5.363,0.002,0.023
3,outright_book,0.047,44.244,-7.449,728.000,0.043,138.015,-7.641,0.004,0.049


### Spread legs

,book,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,turnover,avg_gross_exposure
0,corn_soybean_spread,1.056,194.296,-0.784,290.000,0.502,214.756,-1.193,0.002,0.044
1,corn_wheat_spread,-0.499,-87.092,-2.435,280.000,-0.630,-371.152,-4.581,0.002,0.044
2,soybean_wheat_spread,0.178,52.293,-3.007,325.000,-0.292,-289.940,-5.841,0.002,0.035
3,spread_book,0.249,156.390,-4.681,500.000,-0.240,-453.743,-9.564,0.005,0.069


### Portfolio comparison

,book,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,turnover,avg_gross_exposure
0,outright_book,0.047,44.244,-7.449,728.000,0.043,138.015,-7.641,0.004,0.049
1,spread_book,0.249,156.390,-4.681,500.000,-0.240,-453.743,-9.564,0.005,0.069
2,outright_plus_spreads,0.136,197.840,-9.798,738.000,-0.070,-319.529,-13.082,0.008,0.092


### OOS strategy PnL correlation

,corn,soybean,wheat,spread_book
corn,1.000,0.235,0.302,0.199
soybean,0.235,1.000,0.019,0.298
wheat,0.302,0.019,1.000,0.380
spread_book,0.199,0.298,0.380,1.000


## 3. Named-Period Check

Use the same named market periods as the single-market notebooks and apply them to the combined outright-plus-spread book.


In [4]:
period_check = shared_named_period_check(combined_bt, DD_CAPITAL_USD)
display_table(period_check[["period", "total_pnl", "sharpe", "max_dd_pct", "hit_rate", "active_days", "note"]])

outright_row = portfolio_results.loc[portfolio_results["book"] == "outright_book"].iloc[0]
spread_row = portfolio_results.loc[portfolio_results["book"] == "spread_book"].iloc[0]
combined_row = portfolio_results.loc[portfolio_results["book"] == "outright_plus_spreads"].iloc[0]
raw_cs = raw_corr.loc["CORN", "SOYABEAN"]
strategy_cs = strategy_pnl.loc[trading_index >= OOS_START].corr().loc["corn", "soybean"]

conclusion = f"""
### Conclusion

The raw grain futures are correlated: corn/soybean OOS daily PnL correlation is `{raw_cs:.2f}`. The simple outright score legs are less correlated: corn/soybean strategy PnL correlation is `{strategy_cs:.2f}`.

The common-score outright book is weak on its own: OOS Sharpe `{outright_row['oos_sharpe']:.3f}`, OOS PnL `{outright_row['oos_pnl']:.3f}`.

The spread book is also not strong as a full basket: OOS Sharpe `{spread_row['oos_sharpe']:.3f}`, OOS PnL `{spread_row['oos_pnl']:.3f}`. Corn/soybean is the cleanest individual spread in this simple test, while wheat spreads are less reliable.

The combined book has OOS Sharpe `{combined_row['oos_sharpe']:.3f}` and OOS DD `{combined_row['oos_dd_pct']:.2f}%`. This argues for treating cross-grain spreads as research candidates to select carefully, not as an automatic overlay.
"""
display(Markdown(conclusion))


,period,total_pnl,sharpe,max_dd_pct,hit_rate,active_days,note
0,2010-2011: Russian drought/export ban shock,-39.886,-0.094,-3.351,0.494,241,
1,2012-2013: US drought rally/retrace,157.144,0.533,-3.819,0.451,213,
2,2014: Crimea/Black Sea shock,-46.024,-0.268,-2.394,0.480,75,
3,2014-2017: Low-price abundant supply,397.088,0.280,-8.644,0.487,759,
4,2018-2020: US-China trade war,-96.332,-0.158,-4.034,0.484,370,
5,2019: 2019 prevented planting floods,27.504,0.136,-3.531,0.515,66,
6,2020: COVID demand shock,-191.975,-1.720,-2.905,0.407,81,
7,2020: COVID recovery/China buying,372.133,0.969,-6.336,0.561,132,



### Conclusion

The raw grain futures are correlated: corn/soybean OOS daily PnL correlation is `0.59`. The simple outright score legs are less correlated: corn/soybean strategy PnL correlation is `0.24`.

The common-score outright book is weak on its own: OOS Sharpe `0.047`, OOS PnL `44.244`.

The spread book is also not strong as a full basket: OOS Sharpe `0.249`, OOS PnL `156.390`. Corn/soybean is the cleanest individual spread in this simple test, while wheat spreads are less reliable.

The combined book has OOS Sharpe `0.136` and OOS DD `-9.80%`. This argues for treating cross-grain spreads as research candidates to select carefully, not as an automatic overlay.
